# Build the best Toffoli gate you can

Your team will submit a quantum circuit which implements a Toffoli (controlled-controlled-NOT) gate using only the supplied **native** gate set. The lab will execute it on real hardware and return raw, shot-by-shot outcomes. Your aim is a Toffoli that is as accurate as you can make it, *and the evidence to show it*.

> **The final challenge: prove that your Toffoli gate is the best.** There is no leaderboard and no scoring formula. You decide what "best" should mean, which experiments make the case, and how sure you are. Sections 7 and 8 show what a truth table can and cannot tell you and point you to the tools (tomography and cheaper relatives) that go further. Section 10 describes what the judges look for.

> **Hardware window: 4:00 PM to 12:00 midnight.** The hardware queue opens at **4:00 PM** and closes at **midnight**. The relay refuses every hardware submission sent before 4:00 PM or after midnight, so nothing can be queued early and nothing can be squeezed in late. Until 4:00 PM, all of your work happens on simulators: use that time to finish and verify your circuit and to plan every hardware experiment you intend to run.
>
> **After midnight: the IonQ extension.** For the remaining 12 hours (midnight to noon) you can keep working on the same problem on IonQ's quantum cloud, with your own IonQ API key, in the IonQ extension notebook, which the organizers release at midnight. Watch for the IonQ invitation email during the evening and accept it early.


---

# Background

### What is a qubit?

The fundamental component of a classical computer is a bit. It can take values either 0 or 1. Bits can be operated on with "gates." Some examples include the AND, NOT, and XOR gates. These represent some basic building blocks that can be strung together to create complex programs. In fact, the computer you're reading on is using these bits and gates to present this text on your screen, communicate with the Internet, and many other core functionalities under the hood. 

Quantum computers similarly have a fundamental component: a quantum bit (qubits). Qubits work differently when compared to classical bits. These need not be in a definite state of 0 or 1. One can prepare a qubit to be in a state that has a *probability* of being in a state of 0 or 1. Quantum mechanics is what makes this possible. This is one core feature of quantum computing that sets it apart from classical computing. We can write a qubit's state in the following manner: $$\ket{\psi} = \alpha \ket{0} + \beta \ket{1}, \quad |\alpha|^2 + |\beta|^2 = 1, \alpha, \beta \in \mathbb{C}.$$ 

Here, $\ket \psi$ is the mathematical object (a vector!) that represents the state of qubit (i.e, a quantum state). Notice that $\alpha, \beta$ are complex-number coefficients whose *magnitudes'* squares must sum to 1. The square of the magnitude of $\alpha, \beta$ represent the probability of observing the qubit in state 0 or 1, respectively. Why the magnitude square? To fully capture the properties of quantum mechanics (namely, interference), we require the use of complex numbers. But, probabilities are strictly real numbers. We account for this by using the magnitude squared. 

Notice that when we observe a qubit, all we see is a state 0 or 1. We don't get to observe the probabilities of seeing 0 and 1. The two independent quantum states $\ket{\psi_1} = \frac{1}{\sqrt{2}}\ket 0 + \frac{1}{\sqrt{2}} \ket 1, \ket{\psi_2} = \frac{1}{\sqrt{3}}\ket 0 + \sqrt{\frac{2}{3}} \ket 1$ each have different probabilities of observing the states 0 and 1. Yet, when we measure both, we may observe both qubits being in state 0.

The way we determine the probabilities of a qubit being in each state is (1) prepare the state, (2) measure it, (3) repeat. Say you repeat this 1000 times. Then, for $\ket{\psi_1}$ you should expect half of your measurement outcomes to be 0 and the other half to be 1. For $\ket{\psi_2}$, we should expect 1/3 of all outcomes to be 0 and 2/3 of all outcomes to be 1.

We can extend this to multiple qubits as well. For instance, one can have a three-qubit state $\ket \phi = \frac{1}{\sqrt 2}\ket{000} + \frac{1}{\sqrt 2}\ket{111}$. If I measure the state $\ket \phi$ I have a 50% probability of measuring all three qubits in the state 0 and a 50% probability of all qubits being in state 1. These outcomes are now bitstrings (000 vs 111) instead of just single bits.

Note: Linear algebra is the language of quantum mechanics. Quantum states are vectors. Operations on states are modeled by matrices (or more, formally, linear operators).

### What is a quantum circuit?

As *classical* circuits represent a set of operations on classical bits, *quantum* circuits represent a series of operations on a set of qubits. These are visually depicted using a circuit diagram, like the ones drawn later in this notebook. Each horizontal line represents a qubit. Each block is a quantum gate. Such gates are operations whose effect on a quantum state is represented by a unitary matrix ($U^\dagger = U^{-1}$, where $\dagger$ is conjugate-transpose). 

Building on the discussion in the previous section, simply running a quantum circuit will provide a bitstring outcome. But, we want to know the actual probabilities of observing a bitstring. For this, we need to run the quantum circuit and measure its output state several times. The number of times one re-runs a circuit is referred to as the number of *shots*.

The visual resources below include examples of common quantum gates. Many one-qubit gates are easiest to understand as rotations of a quantum state on the Bloch sphere.

## Setup

Run this once on a local notebook environment. The simulator is only for development; it is not a model of the live device unless a device-specific noise model is supplied.

**Important:** from the root folder of your fork (the folder containing `requirements.txt`), install the tested package versions *and* this repository as an editable package, then start Jupyter:

```bash
python -m pip install -r requirements.txt
jupyter lab
```

Use Python 3.10 or newer (3.12 is what the kit was tested with); a fresh virtual environment (`python -m venv .venv`) is recommended. Make sure the notebook kernel is the same environment you installed into. If you prefer not to install anything locally, open your fork in **GitHub Codespaces** (Code → Codespaces); it builds this environment automatically.

The editable install is what makes `import toffoli_challenge` work even though the notebook and package live in different folders. If you edit a helper module locally, restart the notebook kernel before re-importing it.


In [ ]:
# Environment check. If this cell fails, rerun the installation in the Setup cell above
# using the same Python environment as this notebook's kernel.
import sys
from collections import Counter
from importlib.metadata import version
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import qiskit
from qiskit import QuantumCircuit, qasm2
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram

try:
    import toffoli_challenge  # noqa: F401
except ImportError as error:
    raise ImportError(
        "toffoli_challenge is not installed in this kernel's environment. From the repository "
        "root run:  python -m pip install -r requirements.txt"
    ) from error

TESTED = {"qiskit": "2.1.1", "qiskit-aer": "0.18.0"}
print("Python:", sys.version.split()[0])
for package, tested in TESTED.items():
    installed = version(package)
    note = "" if installed == tested else f"  (kit tested with {tested}; reinstall from requirements.txt if anything breaks)"
    print(f"{package}: {installed}{note}")
print("Environment ready.")


### Visual resources and a first circuit diagram

The following resources complement the discussion above:

- [Qiskit circuit-visualization guide](https://quantum.cloud.ibm.com/docs/en/guides/visualize-circuits)
- [Qiskit Bloch-vector visualization API](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.visualization.plot_bloch_multivector)
- [Qiskit Q-sphere visualization API](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.visualization.plot_state_qsphere)
- [IBM Quantum Learning: quantum circuits](https://quantum.cloud.ibm.com/learning/en/courses/basics-of-quantum-information/quantum-circuits/introduction)

The next cell draws a two-qubit circuit. Read it from left to right: H creates a superposition on the first wire, and the connected dot/X symbol is a controlled-NOT.

In [ ]:
# A first example of a circuit diagram: H followed by a CNOT.
intro_circuit = QuantumCircuit(2)
intro_circuit.h(0)
intro_circuit.cx(0, 1)
display(intro_circuit.draw("mpl"))

## 1. Introduction to Gates: CNOT Gate

A quantum circuit is a sequence of unitary operations on qubits. A one-qubit state may be a superposition

$$|\psi\rangle = \alpha|0\rangle + \beta|1\rangle,\qquad |\alpha|^2+|\beta|^2=1.$$

A measurement returns a classical bit. Repeating the same circuit many times (shots) estimates the outcome probabilities; a single shot is only one sample.

### The controlled-NOT (CNOT) gate

A CNOT has a **control** bit $c$ and a **target** bit $t$. It flips the target iff the control is $1$: $|c,t\rangle\mapsto|c,t\oplus c\rangle$. In Qiskit, `qc.cx(control, target)` applies it.

| Input $|ct\rangle$ | Output |
|:---:|:---:|
| $|00\rangle$ | $|00\rangle$ |
| $|01\rangle$ | $|01\rangle$ |
| $|10\rangle$ | $|11\rangle$ |
| $|11\rangle$ | $|10\rangle$ |

For a general two-qubit input (ordered as control, target),

$$|\psi\rangle=\alpha|00\rangle+\beta|01\rangle+\gamma|10\rangle+\delta|11\rangle,$$

linearity gives

$$\operatorname{CNOT}|\psi\rangle=\alpha|00\rangle+\beta|01\rangle+\gamma|11\rangle+\delta|10\rangle.$$

The amplitudes of the $c=0$ terms remain where they are; the two $c=1$ target amplitudes exchange places. This is why a CNOT acting on $H|0\rangle\otimes|0\rangle$ creates an entangled Bell state.

## 2. Why are circuits run many times?

The next helper runs *simulator-only teaching circuits*. It appends final measurements to a copy of the circuit, prints the complete list of bitstrings (`memory`), and then plots their frequency histogram. These measurement instructions are allowed here, but **never** in a hardware submission.

In [ ]:
def run_and_show_memory(circuit, shots=64, seed=7):
    """Run a teaching circuit, print every sampled bitstring, and plot counts."""
    measured = circuit.copy()
    measured.measure_all()
    result = AerSimulator(seed_simulator=seed).run(
        measured, shots=shots, memory=True
    ).result()
    bitstrings = result.get_memory(measured)
    counts = Counter(bitstrings)
    print(f'Complete shot list ({len(bitstrings)} strings):')
    print(bitstrings)
    print('\nCounts:', dict(sorted(counts.items())))
    figure = plot_histogram(counts, title=f'{shots}-shot outcome histogram')
    display(figure)
    plt.close(figure)
    return bitstrings, counts

basic_circuits = {}

basic_circuits['always 00'] = QuantumCircuit(2)

x_on_first = QuantumCircuit(2)
x_on_first.x(0)
basic_circuits['X on q0'] = x_on_first

bell = QuantumCircuit(2)
bell.h(0)
bell.cx(0, 1)
basic_circuits['Bell-state circuit'] = bell

for name, circuit in basic_circuits.items():
    print(f'\n--- {name} ---')
    display(circuit.draw('mpl'))
    run_and_show_memory(circuit, shots=64)

### Reading the histogram

The height of each bar divided by the number of shots is an empirical probability. For the Bell circuit, ideal quantum mechanics assigns probability $1/2$ to `00` and $1/2$ to `11`; a 64-shot run will generally not return exactly 32 of each. Increasing the number of shots reduces statistical uncertainty, but it cannot undo hardware noise.

Qiskit writes a multi-qubit measurement string with the highest-indexed qubit on the left (`q[n-1] ... q[0]`). The challenge wire convention below deliberately uses this convention so the returned binary integer has the same order as the logical Toffoli bits.

## 3. Problem Statement: Produce the Best Toffoli Gate

The Toffoli, or controlled-controlled-NOT (CCX), has two controls $c_1,c_2$ and a target $t$:

$$|c_1c_2t\rangle\longmapsto |c_1c_2(t\oplus c_1c_2)\rangle.$$

Only when **both** controls equal 1 does the target flip. In some sense, this resembles a 3-qubit, extended version of a CNOT gate.

```text
c₁ ──●──
     │
c₂ ──●──
     │
 t ──⊕──
```

| $c_1c_2t$ input | output |
|:---:|:---:|
| 000 | 000 |
| 001 | 001 |
| 010 | 010 |
| 011 | 011 |
| 100 | 100 |
| 101 | 101 |
| 110 | 111 |
| 111 | 110 |

A high-level Qiskit `ccx` is convenient for describing the desired operation but is **not** an allowed submission instruction. The competition is about replacing it with a short, accurate sequence from the supplied native gates.

In [ ]:
# The target in the challenge's wire convention: controls q[2], q[1]; target q[0].
toffoli_picture = QuantumCircuit(3, name='Toffoli')
toffoli_picture.ccx(2, 1, 0)
display(toffoli_picture.draw('mpl'))


### Starting points for synthesis

Useful references (read critically—many textbook decompositions assume a different native entangling gate):

- [Qiskit transpilation guide](https://quantum.cloud.ibm.com/docs/en/guides/transpile). Note that `transpile(circuit, basis_gates=["rx", "ry", "rz", "rxx"])` compiles directly to this challenge's native gates; Section 7 uses it to give you a baseline to beat.
- [Qiskit unitary-synthesis documentation](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.transpiler.passes.UnitarySynthesis)
- [Qiskit synthesis overview](https://quantum.cloud.ibm.com/docs/en/guides/synthesize-unitary-operators)
- [OpenQASM 2 export in current Qiskit](https://quantum.cloud.ibm.com/docs/en/guides/interoperate-qiskit-qasm2)
- [BQSKit](https://bqskit.readthedocs.io/en/latest/), a numerical synthesis and optimization toolkit.

Background reading on Toffoli gates and trapped-ion compilation:

- D. Maslov, [Basic circuit compilation techniques for an ion-trap quantum machine](https://arxiv.org/abs/1603.07678) (2017): circuit identities for the `rxx` (Mølmer–Sørensen) gate, including Toffoli constructions.
- V. Shende and I. Markov, [On the CNOT-cost of TOFFOLI gates](https://arxiv.org/abs/0803.2316) (2009): lower bounds on how many two-qubit gates a Toffoli needs.
- T. Monz *et al.*, [Realization of the quantum Toffoli gate with trapped ions](https://arxiv.org/abs/0804.0082) (2009), and C. Figgatt *et al.*, [Complete 3-qubit Grover search on a programmable quantum computer](https://arxiv.org/abs/1703.10535) (2017): Toffoli-type gates on real ion-trap hardware.

Keep the *logical target* fixed while exploring alternatives: cancellation of adjacent rotations, identities up to global phase, different control/target conventions, and re-synthesis of short two-qubit blocks are all fair game.


# ** HARDWARE SHOTS ARE LIMITED — VERIFY ON A SIMULATOR FIRST**

**Each team has a total hardware budget of **20,000 shots** on the DQC trapped-ion quantum computer for the entire challenge.** Every device execution consumes this shared budget: retries, alternate decompositions, calibration experiments, and all circuits in an error-mitigation ensemble count.

**Hardware submissions are accepted only from 4:00 PM to 12:00 midnight.** The relay rejects anything sent outside that window. Your final circuit all have to enter the queue before midnight, so leave yourself time for the lab to return results and for a second attempt. Expect uptime from submission time until results are returned.

**Do not use hardware to debug a circuit.** Before submitting, verify its unitary action, wire order, native-gate compliance, and output format on a simulator.

The menu entry in a submission is one of `64`, `128`, `256`, `512`, or `1024` shots. The lab runs each submission for the specified number of shots, from the initial state `|000>`. Choose your shot number wisely—you have a strict budget!

## 4. Real hardware is noisy

An ideal gate applies exactly its intended unitary. But, real quantum computers are noisy. Errors can happen when running quantum gates. Noise channels provide a mathematical representation of how noise affects our quantum states.

This challenge has three simulator stages:

1. **A known, disclosed noise model.** Its parameters are visible and let you practice diagnosing a structured error.
2. **A mystery backend.** It returns samples but does not disclose its model; use probe circuits to form and test hypotheses.
3. **Your own local models.** Build models that stress-test a candidate before any hardware submission.

None of these simulators is a calibration model of the live device. They are development tools for learning how a correct unitary can still produce imperfect samples.

In [ ]:
# 4.1 Known noise model: a compensated RXX overrotation
#
# The model is fully disclosed in toffoli_challenge/known_noise.py:
#   * every rxx(theta) is followed by rxx(+0.2), so its net angle is theta + 0.2;
#   * rx and ry have depolarizing probability 0.001;
#   * readout is asymmetric: 1 -> 0 is more likely than 0 -> 1.
#
# A single small overrotation is hard to see in a histogram. Repeating the gate
# amplifies it: four rxx(pi/2) gates make rxx(2*pi), which is the identity up to a
# global phase, so the ideal output is always 00. Under the known model the four
# offsets add up to 0.8 rad and some shots flip to 11. Since rxx rotations share a
# generator, commanding theta - 0.2 instead of theta cancels the coherent part; the
# remaining stochastic and readout errors cannot be removed by this compensation.
from toffoli_challenge.known_noise import run_with_final_measurements

OVERROTATION = 0.2

def repeated_rxx(angle, repetitions=4):
    circuit = QuantumCircuit(2)
    for _ in range(repetitions):
        circuit.rxx(angle, 0, 1)
    return circuit

ideal_demo = repeated_rxx(np.pi / 2)
ideal_demo.measure_all()
ideal_counts = AerSimulator(seed_simulator=19).run(ideal_demo, shots=1024).result().get_counts()
known_counts = run_with_final_measurements(repeated_rxx(np.pi / 2), shots=1024, seed=19)
compensated_counts = run_with_final_measurements(
    repeated_rxx(np.pi / 2 - OVERROTATION), shots=1024, seed=19
)

print('Ideal counts:              ', ideal_counts)
print('Known-noise counts:        ', known_counts)
print('Known noise, compensated:  ', compensated_counts)
figure = plot_histogram(
    [ideal_counts, known_counts, compensated_counts],
    legend=['ideal', 'known model', 'known model, angle compensated'],
    title='Four rxx(pi/2) gates: ideal, noisy, and compensated',
)
display(figure)
plt.close(figure)


### 4.2 Build and test your own noise models

The disclosed warm-up model is intentionally narrow. Qiskit Aer supports several useful model families:

- **Stochastic gate channels:** depolarizing, Pauli, amplitude damping, phase damping, and thermal-relaxation errors.
- **Coherent errors:** a fixed unwanted unitary such as an RXX overrotation.
- **Readout errors:** a classical assignment matrix applied at measurement.
- **Backend-derived approximations:** a model built from a backend or fake-backend calibration snapshot.

Useful references:

- [Qiskit Aer: building custom noise models](https://qiskit.github.io/qiskit-aer/tutorials/3_building_noise_models.html)
- [Qiskit Aer noise-model API](https://qiskit.github.io/qiskit-aer/apidocs/aer_noise.html)
- [Qiskit Aer: device-backend noise-model simulations](https://qiskit.github.io/qiskit-aer/tutorials/2_device_noise_simulation.html)
- [Qiskit Aer: applying noise to custom unitary gates](https://qiskit.github.io/qiskit-aer/tutorials/4_custom_gate_noise.html)

When comparing decompositions, vary both coherent and stochastic assumptions. A circuit that is best under depolarizing noise need not be best under systematic entangling-angle error or asymmetric readout.

## 5. Warm-up on a simulator

Alongside the known model, your team receives access to an **opaque simulator**. It accepts circuits and returns samples, but the noise model is not disclosed. Use a small number of probe circuits to infer its dominant behavior. Is its largest effect consistent with readout error, one-qubit error, entangling-gate error, coherent overrotation, or something else?

This is a simulator-only warm-up and does **not** consume the real-hardware budget. It is designed to reward experimental reasoning, not just compilation. The mystery backend is remote: the hidden model is not present in this repository.

In [ ]:
# The package contains only a remote client; the hidden model is not in this fork.
# Ask the organizers for MYSTERY_BACKEND_URL and, if required, MYSTERY_BACKEND_TOKEN.
import os
from toffoli_challenge.mystery_backend import mystery_backend

# Example event configuration. Do not commit event credentials to your fork.
# os.environ['MYSTERY_BACKEND_URL'] = 'https://organizer.example/mystery'
# os.environ['MYSTERY_BACKEND_TOKEN'] = 'provided-by-organizers'

def mystery_counts(circuit, shots=256):
    # The remote service adds final measurements itself; candidates stay measurement-free.
    return mystery_backend.run(circuit, shots=shots).result().get_counts()


# Suggested native-gate probes. Add your own, but write down the hypothesis each tests.
probes = {}
probes['prepare |0>'] = QuantumCircuit(1)

one_state = QuantumCircuit(1)
one_state.rx(np.pi, 0)
probes['prepare |1> with RX(pi)'] = one_state

two_rx = QuantumCircuit(1)
two_rx.rx(np.pi, 0)
two_rx.rx(np.pi, 0)
probes['two RX(pi) gates (ideal identity)'] = two_rx

entangling_probe = QuantumCircuit(2)
entangling_probe.ry(np.pi / 2, 0)
entangling_probe.rxx(np.pi / 2, 0, 1)
probes['native RXX entangling probe'] = entangling_probe

if not mystery_backend.url:
    print('MYSTERY_BACKEND_URL is not set yet; skipping the probes. Set it above when the organizers share it.')
else:
    for name, probe in probes.items():
        print(f'\n--- {name} ---')
        print(mystery_counts(probe, shots=256))

## 6. Challenge specification

**Native gates.** Every quantum computer has a few operations it performs directly; everything else is built from them. On this trapped-ion machine the entangling gate is $R_{XX}(\theta)$, an $XX$ rotation. Your circuit may use only [`rx`](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.circuit.library.RXGate), [`ry`](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.circuit.library.RYGate), [`rz`](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.circuit.library.RZGate) and [`rxx`](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.circuit.library.RXXGate). `rz` is done in software, so it is free: use it as much as you like.

**Rules for a submitted circuit** (`validate_submission_circuit` checks them all):

- exactly **three** qubits and no classical bits;
- only `rx`, `ry`, `rz`, `rxx`, with plain numerical angles; no measurements, resets, barriers or other instructions (the lab adds the measurements);
- wires: `q[2] = c₁`, `q[1] = c₂`, `q[0] = t`. The target is `ccx(2, 1, 0)`, and every bitstring reads `c₁c₂t`.

**What the lab does with it.** You pick `N` from `64`, `128`, `256`, `512`, `1024`. The lab starts from `|000>`, runs your circuit exactly as written, measures `N` times, and returns one raw list of bitstrings, for example `['111', '111', '110', ...]`, and nothing else. Files are named `<queueID>_<N>_<teamID>.qasm` (yours) and `.txt` (the result). The lab adds no input preparation. To test input `110`, submit `with_input_preparation(candidate, '110')`, and write down which queue ID prepared which input.

The next cell demonstrates the helpers that check circuits and turn raw lists into numbers (code in `toffoli_challenge/`).

In [ ]:
# Helper demo. Judges may recompute your numbers from the lab's raw files: keep the queue IDs.
from toffoli_challenge import bitstrings_to_counts, truth_table_row, validate_submission_circuit

# This circuit is native-gate compliant but is NOT a Toffoli; it only demonstrates validation.
demo_native_circuit = QuantumCircuit(3)
demo_native_circuit.rx(np.pi / 2, 0)
demo_native_circuit.rz(-np.pi / 4, 1)  # Virtual/software rotation: free on this hardware.
demo_native_circuit.rxx(np.pi / 4, 2, 1)

print('1) Valid native circuit:')
validate_submission_circuit(demo_native_circuit)
display(demo_native_circuit.draw('mpl'))

print('\n2) Invalid circuit: the high-level H instruction is rejected.')
invalid_demo = QuantumCircuit(3)
invalid_demo.h(0)
try:
    validate_submission_circuit(invalid_demo)
except ValueError as error:
    print(error)

print('\n3) Analyze one bare raw list returned for prepared input 110:')
demo_raw_bitstrings = ['111'] * 60 + ['110'] * 4
demo_row = truth_table_row('110', demo_raw_bitstrings)
print(f'Expected output: {demo_row.ideal_output}; success: {demo_row.success_probability:.4%}')
figure = plot_histogram(demo_row.counts, title='Demo raw outcomes for input 110')
display(figure)
plt.close(figure)

print('\n4) Turn a raw bitstring list into counts:')
print(bitstrings_to_counts(demo_raw_bitstrings))

## 7. Develop and verify your candidate locally

Local unitary comparison is a fast way to catch a wrong wire order or a decomposition that differs from Toffoli by more than global phase. It is exact-math verification—not a prediction of device performance.

The next function expects the required convention: candidate `q[2]` and `q[1]` are controls, and `q[0]` is the target. A global phase is accepted because it has no observable effect.

In [ ]:
from toffoli_challenge import compare_to_toffoli, ideal_toffoli_in_submission_order

# This reference circuit is intentionally NOT submission-valid because ccx is high level.
reference_target = ideal_toffoli_in_submission_order()
max_error, process_fidelity, average_gate_fidelity = compare_to_toffoli(reference_target)
print(f'Maximum entrywise error after global-phase alignment: {max_error:.3e}')
print(f'Ideal process fidelity: {process_fidelity:.12f}')
print(f'Ideal average gate fidelity: {average_gate_fidelity:.12f}')
try:
    validate_submission_circuit(reference_target)
except ValueError as error:
    print('As intended, the high-level reference circuit is rejected:')
    print(error)

### A baseline to beat

Qiskit can compile the reference Toffoli straight into the native gate set. The result is a valid, exact submission with five `rxx` gates. Use it as a correctness reference and as the design to beat: the cell below also estimates its truth-table success rate under the known noise model, using the eight-input truth-table experiment explained at the start of Section 8.


In [ ]:
from toffoli_challenge import baseline_native_toffoli
from toffoli_challenge.known_noise import known_noise_truth_table

baseline = baseline_native_toffoli()
validate_submission_circuit(baseline)
print('Native gate counts:', dict(baseline.count_ops()))
max_error, process_fidelity, _ = compare_to_toffoli(baseline)
print(f'Ideal max error: {max_error:.2e}; ideal process fidelity: {process_fidelity:.6f}')
display(baseline.draw('mpl'))

baseline_report = known_noise_truth_table(baseline, shots=1024, seed=7)
print(f'Known-noise truth-table success rate: {baseline_report.mean_success_probability:.4f}')
for input_bits, entry in baseline_report.per_input.items():
    print(f'  {input_bits} -> {entry.ideal_output}: {entry.success_probability:.4f}')


### Strategies worth investigating

There is no single complete mitigation recipe. First distinguish **physical error suppression**—making the circuit itself better—from **estimator mitigation**—post-processing many noisy circuit outcomes into a better estimate. Only the first category improves the raw truth table of a single circuit.

**1. Synthesis, compilation, and coherent-error-aware design**

- Minimize the number and total exposure of entangling `rxx` gates; test distinct exact decompositions, not only gate count.
- Use exact identities and numerical optimization to compress surrounding one-qubit rotations. `rz` is virtual/free, so phase-frame changes are effectively free resources.
- In the disclosed model, each `rxx(θ)` carries a repeatable `+0.2` angle offset. A calibrated angle compensation or a composite/echo-style construction can directly address that coherent component. It will **not** remove the model's depolarization or readout assignment error.
- If the physical model has drift, crosstalk, leakage, or non-Markovian effects, a compensation optimized for a static overrotation can fail. Probe before committing your scarce final shots.

**2. Noise tailoring and suppression**

- Randomized compiling / Pauli twirling can turn coherent buildup into more stochastic-looking noise when an ensemble of logically equivalent circuits is permitted. See [Wallman & Emerson (2016)](https://arxiv.org/abs/1512.01098).
- Dynamical decoupling/echo methods can help idle-time or certain correlated errors, but add gates and are not a generic cure for an entangling-angle error.

**3. Estimator-level error mitigation**

- Readout mitigation estimates and inverts a measured assignment matrix. It can correct *reported probability estimates*, but cannot alter the raw bitstrings already produced.
- Zero-noise extrapolation (ZNE) executes deliberately noise-amplified/folded circuits and extrapolates an **expectation value** toward zero noise. It costs an ensemble of circuits and is not guaranteed to be unbiased. See [Qiskit's mitigation guide](https://quantum.cloud.ibm.com/docs/en/guides/error-mitigation-and-suppression-techniques).
- Probabilistic error cancellation can in principle mitigate calibrated gate noise, but often has severe sampling overhead—usually unattractive under a small shot budget.

**What this means for your case:** a raw $S_{\mathrm{TT}}$ from one circuit measures *the circuit*. A mitigated number measures *the circuit plus your estimator*. Both are fair evidence in this open challenge, but they answer different questions. If you use mitigation, show the raw numbers next to the mitigated ones, say how many extra shots the mitigation cost, and explain what could make the estimator wrong.

Useful broad references: [Endo *et al.* review](https://arxiv.org/abs/2011.01382), [Cai *et al.* review](https://arxiv.org/abs/2210.00921), and [Qiskit's overview](https://quantum.cloud.ibm.com/docs/en/guides/error-mitigation-overview).

## 8. Is it really a Toffoli? From truth tables to tomography

### The natural first test: a truth table

There are $2^3 = 8$ computational-basis inputs, `000` to `111`. A truth-table experiment prepares each one, applies your circuit, measures in the $Z$ basis, and checks whether the output is Toffoli's $T(x)$. That is eight input preparations and eight $Z$-basis measurements (not eight different measurement bases). With `N` shots per input, the truth-table success rate is

$$S_{\mathrm{TT}}=\frac18\sum_{x\in\{0,1\}^3}\frac{\#\{\text{shots with }y=T(x)\}}{N},$$

and it costs $8N$ shots: eight submissions, one per input. Experimental papers routinely report one (for example the iToffoli experiment in [Wei *et al.*, *Nature Physics* 18, 597 (2022)](https://www.nature.com/articles/s41567-022-01590-3)).

### Why it is not enough

A truth table asks one question per input: *starting from $|x\rangle$, how often do I measure $T(x)$?* It measures the eight numbers $|\langle T(x)|U|x\rangle|^2$ and never looks at the **phases** of $U$.

That blind spot is large. Any circuit of the form

$$U = \mathrm{Toffoli}\cdot D,\qquad D=\mathrm{diag}(e^{i\phi_{000}},\dots,e^{i\phi_{111}}),$$

passes every truth-table row perfectly, whatever the eight phases $\phi_x$ are. Only a common (global) phase is physically meaningless. *Relative* phases change what the gate does to superpositions, and a Toffoli inside a larger algorithm (Grover search, arithmetic) depends on them. To see them you must probe superposition inputs and measure in other bases. That is what **state and process tomography**, and their cheaper relatives, do. The rest of this section sets up that problem.

The next cell builds an impostor. It uses only **three** `rxx` gates (the baseline uses five), passes 100% of the ideal truth table, and is still not a Toffoli.

In [ ]:
# A relative-phase "lookalike": perfect truth table, wrong gate.
from qiskit import transpile
from qiskit.quantum_info import Statevector
from toffoli_challenge import TOFFOLI_INPUTS, compare_to_toffoli, with_input_preparation

lookalike = QuantumCircuit(3)
lookalike.ry(np.pi / 4, 0)
lookalike.cx(1, 0)
lookalike.ry(np.pi / 4, 0)
lookalike.cx(2, 0)
lookalike.ry(-np.pi / 4, 0)
lookalike.cx(1, 0)
lookalike.ry(-np.pi / 4, 0)
lookalike = transpile(lookalike, basis_gates=['rx', 'ry', 'rz', 'rxx'], optimization_level=3, seed_transpiler=1)
validate_submission_circuit(lookalike)
print('Native gate counts:', dict(lookalike.count_ops()))

print('\nIdeal (noise-free) truth table:')
for input_bits in TOFFOLI_INPUTS:
    probabilities = Statevector(with_input_preparation(lookalike, input_bits)).probabilities_dict()
    output = max(probabilities, key=probabilities.get)
    print(f'  {input_bits} -> {output}   probability {probabilities[output]:.3f}')

_, lookalike_process_fidelity, _ = compare_to_toffoli(lookalike)
print(f'\nIdeal process fidelity with Toffoli: {lookalike_process_fidelity:.4f}  (a true Toffoli gives 1.0000)')


### Your task: design a verification experiment

`compare_to_toffoli` catches the lookalike instantly, but only because it reads the *ideal* unitary of the circuit you wrote. On hardware you cannot read a unitary; you only get bitstrings. So the question is:

> **Using only what the lab offers, design an experiment that would convince a skeptic that your hardware circuit acts as a Toffoli, phases included, and estimate how close it is.**

**The rules you have to work within**

- The lab starts every run in $|000\rangle$ and measures all three qubits in the $Z$ basis. Nothing else.
- So every input state you want and every measurement basis you want must be built from native gates **inside your submitted circuit**: *prepare → your candidate → basis change → (lab measures Z)*.
- Each distinct circuit is a separate submission of `N` shots, charged against your team's budget, and has to enter the queue between **4:00 PM and midnight**.
- An estimated probability from `N` shots carries a statistical error of about $\sqrt{p(1-p)/N}$.

**Building blocks** (check each one with `Statevector` yourself; signs and conventions are easy to get wrong):

| from $|0\rangle$, prepare | native gate | | to measure in basis | apply before the lab's Z measurement |
|:---:|:---:|---|:---:|:---:|
| $|1\rangle$ | `rx(π)` | | $Z$ | nothing |
| $|+\rangle$ | `ry(π/2)` | | $X$ | `ry(-π/2)` |
| $|-\rangle$ | `ry(-π/2)` | | $Y$ | `rx(π/2)` |
| $|{+i}\rangle$ | `rx(-π/2)` | | | |
| $|{-i}\rangle$ | `rx(π/2)` | | | |

**Questions to work through before 4:00 PM**

1. Describe exactly which circuits pass the truth table perfectly. How much extra information do you need to pin $U$ down up to a global phase?
2. Full **process tomography** of a three-qubit operation needs an informationally complete set of input states and of measurement bases. How many distinct circuits is that (for example $4^3$ inputs × $3^3$ bases)? At 64 shots each, how many shots? Compare that with your 20,000-shot budget and with the 8-hour window.
3. Can you find inputs for which the ideal Toffoli output is still easy to recognize, so that one measurement setting gives a clear pass/fail per shot, just like a truth-table row? Which phases $\phi_x$ does each such test actually constrain? Which tests catch the lookalike above?
4. Is there a cheap pair of *classical* fidelities that together bound the process fidelity? (See Hofmann below.)
5. Your preparation and basis-change gates are noisy and readout is imperfect too. How would you tell a bad Toffoli apart from bad state preparation and measurement (SPAM)?
6. Rehearse your whole plan on the simulators (the disclosed known-noise model and the mystery backend) *before* you spend hardware shots. Does it reject the lookalike? Does it give a believable fidelity for the baseline?

The cell after the resources shows how to assemble and rehearse a single verification circuit. Choosing *which* circuits to run, and how to turn their counts into a conclusion, is up to you.

### Resources

**Tomography**

- M. A. Nielsen and I. L. Chuang, *Quantum Computation and Quantum Information*, Section 8.4 (quantum process tomography): the standard introduction.
- [Qiskit Experiments: state tomography](https://qiskit-community.github.io/qiskit-experiments/manuals/verification/state_tomography.html) and the [`ProcessTomography` API](https://qiskit-community.github.io/qiskit-experiments/stubs/qiskit_experiments.library.tomography.ProcessTomography.html). Useful for generating settings and fitting a reconstruction. It is not installed by default (`pip install qiskit-experiments`; check it supports your Qiskit version), and its circuits use non-native preparation/measurement gates and contain measurements, so you must rewrite them in native gates without measurements before submitting.
- J. A. Smolin, J. M. Gambetta and G. Smith, [Efficient method for computing the maximum-likelihood quantum state from measurements with additive Gaussian noise](https://arxiv.org/abs/1106.5458) (2012): turning noisy counts into a physical state.
- [Qiskit `quantum_info`](https://quantum.cloud.ibm.com/docs/en/api/qiskit/quantum_info): `Statevector`, `DensityMatrix`, `Operator`, `Choi`, `process_fidelity` and `state_fidelity` for checking your analysis on simulated data.
- T. Monz *et al.*, [Realization of the quantum Toffoli gate with trapped ions](https://arxiv.org/abs/0804.0082) (2009): a trapped-ion Toffoli characterized by full process tomography. See how they report it.

**Cheaper than full tomography**

- H. F. Hofmann, [Complementary classical fidelities as an efficient criterion for the evaluation of experimentally realized quantum operations](https://arxiv.org/abs/quant-ph/0411011) (*PRL* 94, 160504, 2005): two truth-table-like measurements in complementary bases bound the process fidelity.
- S. T. Flammia and Y.-K. Liu, [Direct fidelity estimation from few Pauli measurements](https://arxiv.org/abs/1104.4695) (2011), and M. P. da Silva, O. Landon-Cardinal and D. Poulin, [Practical characterization of quantum devices without tomography](https://arxiv.org/abs/1104.3835) (2011): estimate fidelity to a target by sampling a few Pauli settings.

**Separating gate errors from SPAM**

- E. Magesan, J. M. Gambetta and J. Emerson, [Robust randomized benchmarking of quantum processes](https://arxiv.org/abs/1009.3639) (2011), and E. Magesan *et al.*, [Efficient measurement of quantum gate error by interleaved randomized benchmarking](https://arxiv.org/abs/1203.4550) (2012).
- A. Erhard *et al.*, [Characterizing large-scale quantum computers via cycle benchmarking](https://arxiv.org/abs/1902.08543) (2019): developed on trapped ions.
- Gate set tomography, which characterizes preparation, gates and measurement together: D. Greenbaum, [Introduction to quantum gate set tomography](https://arxiv.org/abs/1509.02921) (2015).

Randomized benchmarking of a full three-qubit gate is expensive. Think about what it would and would not tell you here before spending shots on it.

In [ ]:
# Assembling and rehearsing ONE verification circuit: prepare -> candidate -> basis change.
# This example (target prepared in |+>, measured in X) is deliberately weak: it does NOT
# catch the lookalike. Designing a set of settings that does is your job.
from toffoli_challenge.known_noise import run_with_final_measurements


def verification_circuit(candidate, prepare, basis_change):
    # All three parts are native and measurement-free; the lab adds the Z measurement.
    circuit = QuantumCircuit(3)
    circuit.compose(prepare, inplace=True)
    circuit.compose(candidate, inplace=True)
    circuit.compose(basis_change, inplace=True)
    validate_submission_circuit(circuit)
    return circuit


prepare = QuantumCircuit(3)
prepare.ry(np.pi / 2, 0)        # target t = q[0] in |+>, controls in |00>
basis_change = QuantumCircuit(3)
basis_change.ry(-np.pi / 2, 0)  # measure the target in the X basis

for name, candidate in [('baseline', baseline), ('lookalike', lookalike)]:
    test = verification_circuit(candidate, prepare, basis_change)
    ideal = Statevector(test).probabilities_dict()
    noisy = run_with_final_measurements(test, shots=1024, seed=11)
    print(f'{name:9s} ideal P(000) = {ideal.get("000", 0):.3f}   known-noise counts: {dict(sorted(noisy.items()))}')

# Submitting it later is the same as any other circuit (between 4 PM and midnight):
# receipt = submit_circuit(verification_circuit(candidate_circuit, prepare, basis_change), shots=64)


## 9. Submit and retrieve results

Submission infrastructure is intentionally under the hood. You do **not** need a GitLab account, GitLab token, SSH key, command-line Git, or repository URL. The organizer-operated relay receives your measurement-free circuit, assigns a queue ID, checks your remaining hardware-shot budget, and handles the private GitLab workflow.

You only call the black-box functions below:

```python
submit_circuit(candidate_circuit, shots=512)
remaining_hardware_shots()
fetch_my_results()
```

Each submission is run once, exactly as written, starting from `|000>`, and returns one result file.

**The queue is open from 4:00 PM to 12:00 midnight only.** Before 4:00 PM and after midnight `submit_circuit(...)` raises an error from the relay and nothing is charged. `remaining_hardware_shots()` and `fetch_my_results()` work at any time, so you can collect results after the queue closes. After midnight the challenge continues on IonQ's cloud until noon (see *What happens at midnight* below).

The organizer provides your team-specific `TEAM_ID` and `TEAM_SECRET`. Do not commit either value to your fork.

### One-time team configuration

Enter the team-specific values supplied by the organizer. The relay address is built into the `toffoli_challenge` package, so there is nothing else to configure. They authenticate your team to the relay; they are **not** GitLab credentials. Keep `TEAM_SECRET` out of commits, screenshots, and public chat.

In [ ]:
# One-time configuration: TEAM_ID and TEAM_SECRET are supplied privately by the organizer.
from getpass import getpass
from toffoli_challenge import (
    connect_team,
    remaining_hardware_shots,
    submit_circuit,
    submit_qasm_text,
)

TEAM_ID = 'PASTE_YOUR_TEAM_ID_HERE'

if TEAM_ID.startswith('PASTE_'):
    print('Fill in TEAM_ID above (from the organizers), then rerun this cell.')
else:
    TEAM_SECRET = getpass('Paste the team relay secret: ')
    connect_team(TEAM_ID, TEAM_SECRET)

    # Check the current team budget at any time.
    status = remaining_hardware_shots()
    print(f"Charged: {status['charged_hardware_shots']}; remaining: {status['hardware_shots_remaining']}")
    # Hardware submissions are accepted only between 4:00 PM and midnight.
    window = status.get('submission_window', {})
    print('Hardware queue is', 'OPEN' if window.get('open') else 'CLOSED',
          f"(opens {window.get('opens_at')}, closes {window.get('closes_at')})")

# Submit directly from a simulator-verified measurement-free QuantumCircuit:
# receipt = submit_circuit(candidate_circuit, shots=512)
# print(receipt['submission'])          # note receipt['submission']['queue_id']


In [ ]:
# Retrieve only this team's returned raw bitstring-list files.
from toffoli_challenge import (
    TOFFOLI_INPUTS,
    fetch_my_results,
    results_by_queue_id,
    truth_table_from_results,
    with_input_preparation,
)

# Each submission returns one file, <queueID>_<shots>_<teamID>.txt.
# A truth table is eight submissions, one per prepared input. Keep the queue IDs:
# queue_id_by_input = {}
# for input_bits in TOFFOLI_INPUTS:
#     receipt = submit_circuit(with_input_preparation(candidate_circuit, input_bits), shots=64)
#     queue_id_by_input[input_bits] = receipt['submission']['queue_id']
#
# After the lab has run all eight:
# returned = fetch_my_results(destination='returned_results')
# report = truth_table_from_results(returned, queue_id_by_input)
# print(f'Truth-table success rate: {report.mean_success_probability:.4%}')
# for input_bits, entry in report.per_input.items():
#     print(input_bits, '->', entry.ideal_output, f'{entry.success_probability:.4%}')
#
# One submission's raw shots, or everything returned so far:
# print(results_by_queue_id(fetch_my_results(destination=None, queue_id=17)))
# print(list(results_by_queue_id(fetch_my_results(destination=None))))

## What happens at midnight, and what to hand in

**Until midnight:** every lab hardware run must be *in the queue* before 12:00 midnight. Results keep arriving after that, and `fetch_my_results()` works at any time.

**At midnight:**

- the lab queue closes;
- the organizers release the **IonQ extension notebook**, together with instructions for getting it into your repository;
- your team's IonQ project gets its budget.

From then until **noon** you can keep improving and testing your Toffoli on IonQ's **Forte 1** trapped-ion computer, with your own IonQ API key. To be ready:

- accept the IonQ invitation email during the evening;
- check that you can sign in at [cloud.ionq.com](https://cloud.ionq.com) and see your team's project `intro-<your team ID>`;
- the extension notebook walks you through the rest (API key, free simulator, then the QPU).

**Your project submission** (the organizers will announce the deadline):

1. **Your circuit:** the OpenQASM of the Toffoli you claim is best, from `qasm2.dumps(candidate_circuit)`.
2. **Your evidence:** the lab queue IDs and IonQ job IDs behind every hardware number you present.
3. **Your presentation:** about 5 minutes plus about 3 minutes of questions, answering the final challenge below.

## 10. The final challenge: prove that your Toffoli gate is the best

This is the question your team answers at the end, and it is deliberately open. Nobody will plug your numbers into a formula. You present a case, and the judges decide how convincing it is.

**"Best" is yours to define, but you have to defend the definition.** Some questions worth answering explicitly:

- **Best compared to what?** The 5-`rxx` baseline, the other designs you tried, the textbook constructions in Section 3's references. A claim of "best" without a comparison is not a claim.
- **Best by which measure?** Truth-table success rate, phase-sensitive tests, an estimated process fidelity, robustness to the noise you found in Section 5, gate count. Section 8 shows why a perfect truth table alone does not prove you built a Toffoli.
- **How sure are you?** Every probability estimated from `N` shots has a statistical error of about $\sqrt{p(1-p)/N}$. Two designs that differ by less than their error bars are tied until you show otherwise. Hardware also drifts, so a comparison is fairest when the runs are taken close together.
- **On which machine?** The known-noise model, the mystery backend, the lab hardware and (after midnight) IonQ's cloud can disagree. Explain what you think each one told you.
- **What would change your mind?** Name the experiment that could have shown your design is *not* the best, and say whether you ran it.

**What the judges look for**

- **Understanding.** You can explain why your circuit works, why each experiment was worth its shots, and what the data does *not* show. Expect questions, and expect "why?" more than once.
- **Evidence over polish.** Plots of your own data with error bars beat beautiful slides. Every hardware number you present should trace back to a queue ID, so judges can recompute it from the lab's raw files.
- **Honest learning.** Failed ideas, wrong hypotheses and what you changed afterwards count in your favour. A team that understands a modest result beats a team that cannot explain an impressive one.
- **Clear communication.** A short, well-organized story: the question, what you tried, what you measured, what you conclude.

AI assistants are allowed as tools, but the judges are evaluating *your* understanding. If you can't explain a step without the assistant, leave it out.
